# Lecture 16 – Text Wrangling and Regex

### DATA 2201, Fall 2025



### Lecture Outline
- **Text Wrangling** – Cleaning and preparing text data  
- **Canonicalization with Python** – Standardizing text formats  
- **Regular Expressions (Regex)** – Pattern matching fundamentals  
- **Common Regex Syntax** – Key symbols and quantifiers  
- **Regex Examples & Demos** – Practical text extraction exercises
  

In [ ]:
# Required packages
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import zipfile
%matplotlib inline

### Example Dataset for Canonicalization Demonstration

In [ ]:
with open("data/log.txt", 'r') as f:
    for line in f:
        print(line)

## Demo 1: Canonicalization with Basic Python

In [ ]:
with open('data/county_and_state.csv') as f:
    county_and_state = pd.read_csv(f)
    
with open('data/county_and_population.csv') as f:
    county_and_pop = pd.read_csv(f)    

#### Suppose we'd like to join these two tables. 
- Unfortunately, we can't, because the strings representing the county names don't match, as seen below.

In [ ]:
county_and_state

In [ ]:
county_and_pop

 Before we can join them, we'll do what I call **canonicalization**.

**Canonicalization:** A process for converting data that has more than one possible representation into a "standard", "normal", or canonical form (definition via Wikipedia).
- That means making data consistent and standardized, so that different versions of the same thing can be treated as equal

In [ ]:
def canonicalize_county(county_name):
    return (
        county_name
        .lower()               # lower case
        .replace(' ', '')      # remove spaces
        .replace('&', 'and')   # replace &
        .replace('.', '')      # remove dot
        .replace('county', '') # remove county
        .replace('parish', '') # remove parish
    )

In [ ]:
county_and_pop['clean_county'] = county_and_pop['County'].map(canonicalize_county)
county_and_state['clean_county'] = county_and_state['County'].map(canonicalize_county)

display(county_and_pop)  # display outputs even if not last line in cell
county_and_state

In [ ]:
county_and_pop.merge(county_and_state, on='clean_county')

## Demo 2: Processing Data from a Text Log Using Basic Python

In [ ]:
with open('data/log.txt', 'r') as f:
    log_lines = f.readlines()

In [ ]:
log_lines

#### Suppose we want to extract the day, month, year, hour, minutes, seconds, and timezone. 
- Looking at the data, we see that these items are not in a fixed position relative to the beginning of the string.
- That is, slicing by some fixed offset isn't going to work.

In [ ]:
log_lines[0][20:31]

In [ ]:
log_lines[1][20:31]

#### Instead, we'll need to use some more sophisticated thinking. Let's focus on only the first line of the file.

In [ ]:
first = log_lines[0]
first

In [ ]:
pertinent = first.split("[")[1].split(']')[0]
day, month, rest = pertinent.split('/')
year, hour, minute, rest = rest.split(':')
seconds, time_zone = rest.split(' ')
day, month, year, hour, minute, seconds, time_zone

<br/>

---
## Regular Expressions

- A regular expression, or **regex** for short, is a sequence of characters used to **match patterns in strings**.
- Think of regex as a "mini-language" (formally, they are a grammar for descirbing a language).
- Regular expressions are implemented in almost every modern computer language: JavaScript, Java, C/C++, C#, Perl, Ruby, R, etc.  In Python, we will use the module `re`. 

- **Pros** They are very powerful and are widely used (virtually every programming language has a module for working with them).

- **Cons** They can be hard to read and have many different "dialects"

#### First, import the `re` module.

In [ ]:
import re

## Regular Expression Syntax


| Regex syntex  |  Meaning   |
|:--------------|:-----------|
| [abc]         | a single character of: a, b, or c  | 
| [^abc]        | none of these characters | 
| [a-z]         | any character in this range |
| [^a-z]        | none of the characters in this range | 
| .             | any single character | 
| \\.           | a period  | 
| [123]         | matches only 1 or 2 or 3 | 
| [0-9]         | matches numbers 0 to 9 | 
| \d            | any digit | 
| \D            | any non-digit character | 
| \w            | any alphanumeric character | 
| \W            | any non-alphanumeric character | 
| \s            | white space | 
| \S            | non-white space | 
| *             | zero or more repetitions | 
| +             | one or more repetitions |
| {m}           | m repetitions | 
| {m,n}         | m - n repetitions | 

## Demo 3- Match North American Phone Number

Let's work to match a North American phone number with a regular expression, e.g., 555-123-4567

In [ ]:
pnum = '555-123-4567'

We will start with using the function `re.match()`

The syntax of the function is `re.match(pattern, string, flags=0)`

In [ ]:
match_obj = re.match('555-123-4567', pnum)

if match_obj: 
  print("The pattern matched")
else: 
  print("The pattern did not match")

In [ ]:
match_obj.group()

This match is a string literal, we match the string directly in the target text. 

Usually, we want to match many possible inputs, what if the phone number was 906-555-1234 or 906-555-9876 or ...

### Match Digits with a Character Class

Let's change the regular expression to match all numbers in the phone number at once. 



In [ ]:
m = re.match('[0123456789]', pnum)
m.group()

In [ ]:
m = re.match('[13579]', pnum)
m.group()

In [ ]:
m = re.match('[13579][13579]', pnum)
m.group()

To match any 10-digit, North American phone number with hyphens, you could do the following: 

In [ ]:
m = re.match('[0-9][0-9][0-9]-[0-9][0-9][0-9]-[0-9][0-9][0-9][0-9]', pnum)
m.group()

### Using a Character Shorthand

We can use the shorthand `\d` to match digits just like [0-9]. 

In [ ]:
m = re.match(r'\d\d\d-\d\d\d-\d\d\d\d', pnum)
m.group()

### Extraction with Regex

Python `re.findall`

In [ ]:
text = "My social security number is 123-45-6789 bro, or actually maybe it’s 321-45-6789.";
pattern = r"[0-9]{3}-[0-9]{2}-[0-9]{4}"
re.findall(pattern, text)  # ['123-45-6789', '321-45-6789']

### Demo 4: Capturing Groups, Back References, and Quantifiers

Let's now match just a portion of the phone number using a *capturing group*.  To create a capturing group enclose a `\d` in parentheses to place it in a group, then follow it with a `\1` to backreference what was captured: 

In [ ]:
m = re.match(r'(\d)\d\1', '707')
m.group()

In the above expression, the `\1` refers back to what was captured in the group enclosed by parentheses.  As a result, this regular expression matches the prefix `707`.  Here is how: 

* `(\d)` matches the first digit and captures it (the number 7)
* `\d` matches the next digit (the number 0) 
* `\1` references the captured digit (the number 7)  

### Using Quantifiers

Here is another way to match a phone number with a different syntax.

In [ ]:
m = re.match(r'\d{3}-?\d{3}-?\d{4}', pnum)
m.group()

The numbers in the curly braces `{` `}` tell the regex processor *exactly* how many occurances of those digits to look for.  The braces with numbers are a kind of *quantifier*.  

The question mark (`?`) is another kind of quantifier.  It follows the hyphen in the regular expression and means the hyphen is optional.  That is, there can be zero or one occurrence of the hyphen.  Other quantifiers exists: the `+` plus sign means "one or more" or the `*` asterisk means "zero or more". 

Test the above regex pattern on other phone number styles.

Let's look at an even more concise (but more complicated regex pattern).

In [ ]:
m = re.match(r'(\d{3,4}[.-]?)+', pnum)
m.group()

We can break this down: 

* `(` opens a capturing group 
* `\` start character shorthand 
* `d` end character shorthand (match any digit) 
* `{` open quantifier 
* `3` minimum quantity to match 
* `,` separate quantities 
* `4` maximum quantity to match 
* `}` close quantifier 
* `[` open character class 
* `.` dot or period 
* `-` literal character to match - hyphen 
* `]` close character class 
* `?` zero or one quantifier 
* `)` close capturing group 
* `+` one or more quantifier 

This works, but would also match other groups of 3 or 4 digits.

In [ ]:
m = re.match(r'(\d{3,4}[.-]?)+', pnum)
m.group()

### Match Arbitrary Number of Characters 

The Kleene Star `*` and Kleene Plus `+` allows to represent **0 or more** and **1 or more** of the character that it follows respectively. 

In the example, let's try to match the first three strings but skip the fourth. 

In [ ]:
ex_strs = ("aaaabcc", "aabbbc", "aaccc", "a")
for elem in ex_strs:
    #print (elem)
    m = re.match("a+b+c+", elem)
    if m: 
        print (elem + "\t" + m.group())
    else:
        print (elem + "\t" + "No match")

### Optional Characters 

Another common quantifier is the `?` character, which denotes optionality. 

This allows you to match either zero or one of the preceding character or group. 

In [ ]:
ex_strs = ("abc", "abcd", "ac", "ad", "bc")
for elem in ex_strs:
    #print (elem)
    m = re.match("ab?c", elem)
    if m: 
        print (elem + "\t" + m.group())
    else:
        print (elem + "\t" + "No match")

### Useful tool writing and testing regular expressions


- A useful tool for designing regular expressions is [regex101.com](https://regex101.com).  Choose the Python “flavor” in the left sidebar

- You use it heavily during lab/homework practice; you should have it open as we work through examples. 

#### Example 1

Write a regular expression that matches `'moon'`, `'moooon'`, etc.
- First, think about how to match strings with any even number of `'o'`s, including zero `'o'`s (i.e. `'mn'`).
- Then, think about how to match only strings with a **positive even** number of `'o'`s.


Try It! [https://regex101.com/r/8tkQ23/1](https://regex101.com/r/8tkQ23/1)

<details>
    <summary>
        Click here to see the answer after you've tried it yourself at <a href='https://regex101.com'>regex101.com</a>
    </summary>
    `m(oo)*n`
</details>

#### Example 2

Write a regular expression that matches `'moon'`, `'moooon'`, `'muun'`, `'muuuuun'` etc.

It should match any strings with a **positive even** number of `'o'`s or a **positvie even** number of `'u'`s in the middle.


Try It! [https://regex101.com/r/kJpHeZ/1](https://regex101.com/r/kJpHeZ/1)

<details>
    <summary>
        Click here to see the answer after you've tried it yourself at <a href='https://regex101.com'>regex101.com</a>
    </summary>
    `m(uu(uu)*|oo(oo)*)n` 
</details>

# Practice Homework – Work with a Classmate

### Matches and Non-match cases

For some problems, we want to match certain strings but also not match others. 

Below are a couple lines, where we only want to match the first three strings, but not the last three strings.

* fan
* man 
* can 
* dan 
* ran
* pan 

In [ ]:
ex_strs = ("fan", "man", "can", "dan", "ran", "pan")
for elem in ex_strs:
    m = re.match("[fmc]an", elem)
    if m:
        print(m.group())

Or, alternatively ...

In [ ]:
for elem in ex_strs:
    m = re.match("[^drp]an", elem)
    if m: 
        print (m.group())

Let's now us this in the phone number example. 

In [ ]:
m = re.match(r'(\d{3}[.-]?){2}\d{4}', pnum)
if m:
    print(m.group())
else:
    print("No match")

## Demo 2 Revisit: Text Log Processing using Regex

Python version:

In [ ]:
line = log_lines[0]
line = '169.237.46.168 - - [26/Jan/2014:10:47:58 -0800] "GET /stat141/Winter04/ HTTP/1.1"'
display(line)
pattern = r'\[(\d+)\/(\w+)\/(\d+):(\d+):(\d+):(\d+) (.+)\]'
re.findall(pattern, line)


In [ ]:
# beyond the scope of lecture, but left here for your interest
day, month, year, hour, minute, second, time_zone = re.search(pattern, line).groups()
day, month, year, hour, minute, second, time_zone


#### Pandas version:

In [ ]:
df = pd.DataFrame(log_lines, columns=['Log'])
df

Option 1: `Series.str.findall`

In [ ]:
pattern = r'\[(\d+)\/(\w+)\/(\d+):(\d+):(\d+):(\d+) (.+)\]'
df['Log'].str.findall(pattern)

#### Option 2: `Series.str.extractall`

In [ ]:
df['Log'].str.extractall(pattern)

Wrangling either of these two DataFrames into a nice format (like below) is left as an exercise for you! You will do a related problem on the homework.


||Day|Month|Year|Hour|Minute|Second|Time Zone|
|---|---|---|---|---|---|---|---|
|0|26|Jan|2014|10|47|58|-0800|
|1|2|Feb|2005|17|23|6|-0800|
|2|3|Feb|2006|10|18|37|-0800|


<br/><br/>
<br/>

---

## Real World Example: Restaurant Data (Text Wrangling & Regular Expression)

In this example, we will show how regexes can allow us to track quantitative data across categories defined by the appearance of various text fields.

In this example we'll see how the presence of certain keywords can affect quantitative data:

> **How do restaurant health scores vary as a function of the number of violations that mention a particular keyword?** 
> <br/>
> (e.g., unclean surfaces, vermin, permits, etc.)

In [ ]:
vio = pd.read_csv('data/violations.csv', header=0, names=['bid', 'date', 'desc'])
desc = vio['desc']
vio.head()

In [ ]:
counts = desc.value_counts()
counts.shape

That's a lot of different descriptions!! Can we **canonicalize** at all? Let's explore two sets of 10 rows.

In [ ]:
counts[:10]

In [ ]:
# Hmmm...
counts[50:60]

In [ ]:
# Use regular expressions to cut out the extra info in square braces.
vio['clean_desc'] = (vio['desc']
             .str.replace('\s*\[.*\]$', '', regex=True)
             .str.strip()       # removes leading/trailing whitespace
             .str.lower())
vio.head()

In [ ]:
# canonicalizing definitely helped
vio['clean_desc'].value_counts().shape

In [ ]:
vio['clean_desc'].value_counts().tail() 

Remember our research question:

> **How do restaurant health scores vary as a function of the number of violations that mention a particular keyword?** 
> <br/>
> (e.g., unclean surfaces, vermin, permits, etc.)

<br/>

Below, we use regular expressions and `df.assign()` ([documentation](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.assign.html?highlight=assign#pandas.DataFrame.assign)) to **method chain** our creation of new boolean features, one per keyword.

In [ ]:
# use regular expressions to assign new features for the presence of various keywords
# regex metacharacter | 
with_features = (vio
 .assign(is_unclean     = vio['clean_desc'].str.contains('clean|sanit'))
 .assign(is_high_risk = vio['clean_desc'].str.contains('high risk'))
 .assign(is_vermin    = vio['clean_desc'].str.contains('vermin'))
 .assign(is_surface   = vio['clean_desc'].str.contains('wall|ceiling|floor|surface'))
 .assign(is_human     = vio['clean_desc'].str.contains('hand|glove|hair|nail'))
 .assign(is_permit    = vio['clean_desc'].str.contains('permit|certif'))
)
with_features.head()

# See You Next Lecture!